# Practica 2.1
# Equipo los Morgue
# 7BM1

- García Martínez Carla Edith 
- Mena Fernández Sofia 
- Suárez Ramírez Gustavo 
- Varillas Figueroa Enrique Uriel 
- García Carrizoza André

# 1. Crear las carpetas del proyecto


In [ ]:
# Importamos la clase Path del módulo nativo pathlib. 
# Esto nos permite manejar y manipular rutas de archivos y directorios 
# de forma orientada a objetos (independientemente de si es Windows, Mac o Linux).
from pathlib import Path

# Definimos la ruta absoluta de la carpeta principal del proyecto.
# Path.cwd() obtiene el directorio de trabajo actual (Current Working Directory).
# El operador '/' está sobrecargado en pathlib para concatenar carpetas fácilmente.
carpeta_proyecto = Path.cwd() / "Reconocimiento_Locutor"  

# Definimos las rutas para las subcarpetas específicas del proyecto.
# A partir de la carpeta principal, creamos una para los datos y otra para los resultados.
carpeta_datos = carpeta_proyecto / "datos"  
carpeta_resultados = carpeta_proyecto / "resultados"  

# Ejecutamos la creación física de los directorios en el disco duro.
# El método .mkdir() crea la carpeta.
# parents=True: Asegura que si una carpeta padre en la ruta no existe, también la cree.
# exist_ok=True: Evita que el código lance un error (FileExistsError) si la carpeta ya había sido creada antes.
carpeta_proyecto.mkdir(parents=True, exist_ok=True)  
carpeta_datos.mkdir(parents=True, exist_ok=True)  
carpeta_resultados.mkdir(parents=True, exist_ok=True)  

# Imprimimos en la salida estándar (consola) las rutas finales.
# Esto sirve como validación visual para asegurar que las variables apuntan a donde esperamos.
print("Carpeta del proyecto:", carpeta_proyecto)
print("Carpeta de datos:", carpeta_datos)
print("Carpeta de resultados:", carpeta_resultados)

# 2. Descargar el subconjunto de LibriSpeech dev-clean

In [ ]:
# Importamos el módulo urllib.request, que pertenece a la biblioteca estándar de Python.
# Este módulo contiene funciones y clases que nos permiten abrir y leer URLs,
# lo cual es ideal para realizar descargas de archivos desde internet.
import urllib.request

# Definimos la URL directa desde donde vamos a descargar nuestro conjunto de datos de audio.
url_descarga = ("https://openslr.elda.org/resources/12/dev-clean.tar.gz")

# Definimos las rutas de destino utilizando la variable 'carpeta_datos' (creada en la celda anterior).
# archivo_comprimido: Será el nombre y ruta final de nuestro archivo una vez descargado completamente.
archivo_comprimido = carpeta_datos / "dev-clean.tar.gz"  
# archivo_temporal: Archivo intermedio que se usará mientras los datos se están bajando.
# Esto es una buena práctica para evitar tener un archivo corrupto si la descarga se interrumpe.
archivo_temporal = carpeta_datos / "dev-clean.tar.gz.part" 

# Comprobamos si el archivo final ya existe en nuestro disco duro mediante el método .exists() de pathlib.
if archivo_comprimido.exists():
    # Si existe, evitamos volver a descargarlo (ahorrando tiempo y ancho de banda).
    print("El archivo ya existe:", archivo_comprimido) 
else:
    # Si no existe, comenzamos el proceso de descarga.
    print("Descargando LibriSpeech dev-clean...")

    # Utilizamos un bloque try-except para manejar posibles errores de conexión o escritura.
    try:
        # urllib.request.urlretrieve obtiene el contenido de la URL y lo guarda directamente en disco.
        # Aquí lo guardamos primero bajo el nombre temporal (.part).
        urllib.request.urlretrieve(url_descarga, archivo_temporal) 

        # Una vez que la descarga de urlretrieve finaliza con éxito, 
        # reemplazamos/renombramos el archivo temporal para que adquiera su nombre definitivo.
        archivo_temporal.replace(archivo_comprimido) 
        print("Descarga finalizada.")

    # Si ocurre cualquier excepción (error de red, tiempo de espera, disco lleno, etc.).
    except Exception:
        print("La descarga no terminó")
        # El comando 'raise' relanza el error original para que la ejecución se detenga 
        # y podamos ver en la consola exactamente qué falló.
        raise  

# Una vez asegurada la existencia del archivo (ya sea bajado ahora o previamente),
# usamos .stat() que devuelve información del sistema sobre el archivo.
# .st_size extrae el peso exacto del archivo en bytes.
tamano_bytes = archivo_comprimido.stat().st_size  

# Convertimos los bytes a Megabytes (MB) para que sea humanamente legible,
# dividiendo la cantidad de bytes entre 1,000,000.
tamano_nb = tamano_bytes / 1_000_000  

# Imprimimos la ruta del archivo y su tamaño calculado.
print("Archivo:", archivo_comprimido)
# Utilizamos f-strings para formatear el texto. 
# {tamano_nb:.2f} indica que queremos mostrar la variable flotante con solo 2 decimales.
print(f"Tamaño: {tamano_nb:.2f} MB")

# 3. Verificar la integridad del archivo descargado

In [ ]:
# Se importa la biblioteca hashlib para aplicar algoritmos de hash seguros (como MD5).
# Esto nos servirá para crear una "huella digital" de nuestro archivo descargado.
import hashlib 
# Se importa la biblioteca urllib.request para abrir URLs y leer datos directamente desde internet.
import urllib.request 

# Se define la URL que contiene un archivo de texto con las firmas (hashes) oficiales del dataset.
url_verificacion = "https://openslr.elda.org/resources/12/md5sum.txt"  

# Usamos el bloque 'with' para abrir la URL de manera segura, asegurando que la conexión se cierre al terminar.
# Se añade un timeout de 30 segundos para evitar que el programa se quede congelado si no hay respuesta del servidor.
with urllib.request.urlopen(url_verificacion, timeout=30) as respuesta: 
    # Leemos la respuesta HTTP cruda en bytes.
    contenido = respuesta.read()

# Se decodifica el contenido en bytes a una cadena de texto (string) utilizando codificación UTF-8, 
# que es el estándar para texto en internet.
texto_verificacion = contenido.decode("utf-8")  

# Inicializamos la variable 'md5_esperado' como 'None'. 
# Aquí guardaremos el hash oficial una vez que lo encontremos en el texto.
md5_esperado = None 

# Iteramos sobre el texto descargado línea por línea.
# El método .splitlines() separa el texto gigantesco en una lista de líneas individuales.
for linea in texto_verificacion.splitlines(): 
    # .split() divide la línea en una lista usando los espacios en blanco como separador.
    # Usualmente las líneas tienen el formato: "hash_md5 nombre_del_archivo"
    campos = linea.split()  

    # Verificamos que la línea realmente contenga dos elementos (el hash y el nombre).
    # Esto evita errores si el archivo de texto tiene líneas vacías o mal formateadas.
    if len(campos) != 2:  
        continue  # Si no tiene 2 campos, saltamos esta iteración y pasamos a la siguiente línea.

    # Extraemos el nombre del archivo (segundo campo en la lista).
    # .lstrip("*") elimina el asterisco inicial si es que el archivo de verificación lo incluye.
    nombre_archivo = campos[1].lstrip("*")  
    # Si la ruta tiene directorios (ej. carpeta/archivo.tar.gz), nos quedamos solo con el nombre del archivo.
    nombre_archivo = nombre_archivo.split("/")[-1]  

    # Comparamos el nombre que acabamos de leer con el nombre de nuestro archivo local.
    # archivo_comprimido.name extrae solo el nombre del archivo de la ruta en pathlib (dev-clean.tar.gz).
    if nombre_archivo == archivo_comprimido.name:  
        # Si coinciden, extraemos el primer campo (el hash oficial) y lo convertimos a minúsculas por seguridad.
        md5_esperado = campos[0].lower()  
        # Como ya encontramos lo que buscábamos, rompemos el ciclo 'for' para no gastar más tiempo de procesamiento.
        break  
    
# Validamos si logramos encontrar el hash. Si sigue siendo 'None', significa que nuestro archivo 
# no estaba listado en el documento oficial, por lo que lanzamos un error que detiene el código.
if md5_esperado is None:  
    raise ValueError("No se encontró la suma de comprobación de dev-clean.")

# Creamos un objeto 'verificador' con el algoritmo MD5 para calcular el hash de nuestro archivo local.
verificador = hashlib.md5()  
# Definimos un tamaño de bloque de 1 Megabyte (1024 bytes * 1024 bytes).
# Es crucial leer el archivo en trozos (bloques) para evitar cargar todo de golpe en la memoria RAM.
tamano_bloque = 1024*1024  

# Abrimos el archivo comprimido que descargamos en modo "rb" (read binary / lectura binaria).
# Los archivos como imágenes, videos o comprimidos deben leerse en binario, no como texto.
with archivo_comprimido.open("rb") as archivo: 
    # Iniciamos un bucle infinito para leer el archivo por fragmentos.
    while True:  
        # Leemos exactamente 1 Megabyte del archivo.
        bloque = archivo.read(tamano_bloque) 

        # Si 'bloque' está vacío (False), significa que llegamos al final del archivo.
        if not bloque:  
            break  # Salimos del bucle infinito.

        # Alimentamos el objeto verificador con el fragmento leído.
        # Esto va actualizando el cálculo del hash en tiempo real sin desbordar la memoria.
        verificador.update(bloque)  

# Una vez que procesamos todo el archivo, extraemos el resultado final del hash.
# .hexdigest() convierte ese resultado binario en una cadena de texto hexadecimal legible.
md5_calculado = verificador.hexdigest() 

# Mostramos ambos hashes en pantalla para comparación visual.
print("MD5 esperado: ", md5_esperado)
print("MD5 calculado: ", md5_calculado)

# Creamos una variable booleana que será 'True' si ambos hashes son exactamente iguales, o 'False' si difieren.
integridad_correcta = md5_calculado == md5_esperado

# Si la variable es 'False', la integridad falló (el archivo está corrupto o incompleto).
# Lanzamos una excepción para abortar la ejecución, evitando extraer basura.
if not integridad_correcta:
    raise ValueError("La verificación falló. No extraigas el archivo todavía")

# Si el código llega a esta línea sin lanzar errores, la descarga es perfecta.
print("Verificación correcta. El archivo está listo para extraerse.")

# 4. Extraer el conjunto de datos LibriSpeech dev-clean

In [ ]:
# Se importa el módulo nativo tarfile, diseñado para leer y escribir archivos 
# empaquetados (.tar) y comprimidos (como .tar.gz o .tgz).
import tarfile 

# Se realiza una validación de seguridad inicial. Depende de la variable booleana 
# 'integridad_correcta' creada en la celda anterior. Si el archivo descargado 
# estaba corrupto, el programa se aborta aquí antes de intentar extraer basura.
if not integridad_correcta:
    raise ValueError("El archivo debe pasar la verificación antes de extraerse.")

# Definimos la ruta específica donde esperamos encontrar los datos de audio 
# una vez que la extracción finalice correctamente.
carpeta_corpus = carpeta_datos / "LibriSpeech" / "dev-clean"

# Usamos .resolve() para obtener la ruta absoluta y real del directorio de datos.
# Esto es muy importante como medida de seguridad (para evitar el ataque "Zip Slip")
# asegurando que los archivos no se extraigan fuera de esta carpeta base.
destino_base = carpeta_datos.resolve()

# Inicializamos un contador para saber cuántos archivos ya fueron extraídos previamente.
archivos_existentes = 0
# Creamos una lista vacía donde almacenaremos únicamente los archivos que faltan por extraer.
miembros_pendientes = []

# Abrimos el archivo comprimido usando el bloque 'with' (para que se cierre automáticamente).
# mode="r:gz" indica que vamos a Leer (r) un archivo comprimido con Gzip (gz).
with tarfile.open(archivo_comprimido, mode= "r:gz") as archivo:
    # Obtenemos la lista completa de todo lo que contiene el archivo comprimido 
    # (carpetas, subcarpetas, archivos de texto, audios, etc.).
    miembros = archivo.getmembers()

    # Iniciamos un bucle para revisar, uno por uno, cada elemento dentro del archivo.
    for miembro in miembros:
        # Construimos la ruta donde este elemento específico debería guardarse en nuestro disco duro
        # y calculamos su ruta absoluta con .resolve().
        ruta_destino = (carpeta_datos / miembro.name).resolve()

        # VALIDACIÓN DE SEGURIDAD (Prevención de Path Traversal):
        # Verificamos que la ruta de destino planeada esté estrictamente contenida dentro 
        # de nuestra carpeta 'destino_base'. Esto bloquea archivos maliciosos que tengan 
        # nombres como "../../../etc/password".
        if destino_base not in ruta_destino.parents:
            raise ValueError(f"Ruta de extracción no permitida:  {miembro.name}")

        # Si el elemento que estamos revisando es un directorio (carpeta):
        if miembro.isdir():
            # Creamos la carpeta físicamente en el disco duro.
            ruta_destino.mkdir(parents = True, exist_ok = True)
            continue # Saltamos a la siguiente iteración del bucle, ya que las carpetas no se extraen como archivos.

        # Si el elemento no es una carpeta, validamos que sí sea un archivo normal (isfile).
        # Esto filtra enlaces simbólicos u otros elementos raros que no nos interesan.
        if not miembro.isfile():
            raise ValueError(f"Tipo de archivo no permitido: {miembro.name}")

        # Comprobamos si el archivo ya existe físicamente en nuestro disco duro (de una extracción pasada).
        if ruta_destino.exists():
            # Si existe, consultamos su tamaño actual en bytes en nuestro disco duro.
            tamano_actual = ruta_destino.stat().st_size

            # Lo comparamos con el tamaño que el archivo declara tener dentro del .tar.gz (miembro.size).
            # Si los tamaños no coinciden, el archivo existente está incompleto o corrupto.
            if tamano_actual != miembro.size:
                raise ValueError(f"El tamaño del archivo no coincide: {ruta_destino}")

            # Si el tamaño coincide, asumimos que el archivo ya está extraído correctamente y aumentamos el contador.
            archivos_existentes += 1
        
        else:
            # Si el archivo NO existe en nuestro disco duro, lo añadimos a la lista de pendientes por extraer.
            miembros_pendientes.append(miembro)

    # Una vez revisados todos los elementos, comprobamos si la lista de pendientes tiene algo.
    if miembros_pendientes:
        print("Extratendo los archivos faltantes...")

        # Ordenamos la extracción masiva, pero pasándole ÚNICAMENTE la lista de los archivos 
        # que nos faltan (miembros_pendientes). Esto ahorra muchísimo tiempo si el proceso 
        # se había interrumpido antes y lo volvemos a ejecutar.
        archivo.extractall(
            path= carpeta_datos,
            members = miembros_pendientes)

# Finalizada la extracción, verificamos que la estructura principal de carpetas del dataset 
# (LibriSpeech/dev-clean) realmente exista y sea un directorio válido.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontró la carpeta del corpus.")

# Imprimimos un resumen en consola de lo que el algoritmo hizo.
print("Archivos existentes con tamaño correcto:", archivos_existentes)
print("Archivos extraídos: ", len(miembros_pendientes))
print("Comprobación de extracción finalizada.")

# 5. Contar los locutores y las grabaciones

In [ ]:
# Verificamos nuevamente que la carpeta del corpus exista y sea un directorio válido.
# Si no existe, lanzamos un error y detenemos la ejecución.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontó la carpeta del corpus")

# Creamos una lista vacía para almacenar las rutas de las carpetas correspondientes a cada locutor.
carpetas_locutores = []

# Iteramos sobre todos los elementos dentro de la carpeta del corpus usando .iterdir()
for carpeta in carpeta_corpus.iterdir():
    # Filtramos: solo nos interesan los elementos que sean directorios (carpetas) 
    # Y cuyo nombre esté compuesto únicamente por números (.isdigit()). 
    # En LibriSpeech, cada locutor tiene una carpeta nombrada con su ID numérico.
    if carpeta.is_dir() and carpeta.name.isdigit():
        carpetas_locutores.append(carpeta)

# Definimos una función auxiliar que toma la ruta de una carpeta y devuelve su nombre 
# convertido a un número entero. Esto nos servirá para ordenar las carpetas numéricamente.
def obtener_identificador(carpeta):
    return int(carpeta.name)

# Ordenamos la lista de carpetas de locutores utilizando la función anterior como criterio (key).
# Así, la carpeta "2" irá antes que la "10", en lugar de ordenarse alfabéticamente.
carpetas_locutores.sort(key = obtener_identificador)

# Inicializamos una lista para guardar un diccionario con la información de cada locutor.
resumen_locutores = []
# Inicializamos un contador global para saber cuántas grabaciones hay en total en todo el dataset.
total_grabaciones = 0

# Iniciamos un bucle para procesar cada carpeta de locutor (ya ordenadas).
for carpeta_locutor in carpetas_locutores:
    # Usamos .rglob("*.flac") (recursive glob) para buscar todos los archivos que terminen en .flac 
    # dentro de la carpeta del locutor y en todas sus posibles subcarpetas.
    # Convertimos el resultado a una lista para poder contar sus elementos.
    archivos_locutor = list(carpeta_locutor.rglob("*.flac"))
    
    # Obtenemos la cantidad de grabaciones (archivos .flac) que tiene este locutor específico.
    cantidad_grabaciones = len(archivos_locutor)

    # Creamos un diccionario llamado 'registro' para organizar la información de este locutor.
    registro ={
        "locutor": carpeta_locutor.name, # El ID del locutor (nombre de la carpeta)
        "grabaciones": cantidad_grabaciones, # Cuántos audios tiene
    }

    # Agregamos este diccionario a nuestra lista de resumen general.
    resumen_locutores.append(registro)
    # Sumamos las grabaciones de este locutor al contador global.
    total_grabaciones += cantidad_grabaciones

# Obtenemos la cantidad total de locutores basándonos en cuántas carpetas encontramos.
numero_locutores = len(carpetas_locutores)

# Comenzamos a imprimir un reporte en la consola.
print("Número de locutores:", numero_locutores)
# Utilizamos f-strings con formato de alineación para crear el encabezado de una tabla.
# '<12' alinea a la izquierda ocupando 12 espacios, y '>12' alinea a la derecha ocupando 12 espacios.
print(f"{'Locutor':<12}{'Grabaciones':>12}")
# Imprimimos una línea de puntos para separar el encabezado de los datos.
print("."*24)

# Recorremos la lista de diccionarios que creamos anteriormente para imprimir cada fila.
for registro in resumen_locutores:
    identificador = registro["locutor"]
    cantidad = registro["grabaciones"]

    # Imprimimos los datos de cada locutor aplicando el mismo formato de columnas que en el encabezado.
    print(f"{identificador:<12}{cantidad:>12}")

# Validaciones finales de consistencia de los datos:
# El dataset dev-clean de LibriSpeech debe tener exactamente 40 locutores.
# Si encontramos un número diferente, levantamos un error indicando que los datos podrían estar incompletos.
if numero_locutores != 40:
    raise ValueError(f"Se esperaban 40 locutores, pero no se encontraron {numero_locutores}.")

# Si el total de grabaciones es cero, significa que no se extrajeron los audios correctamente.
if total_grabaciones == 0:
    raise ValueError("No se encotraron grabaciones FLAC.")

# 6. Crear el inventario de metadatos de los audios 

In [ ]:
# Importamos pandas, la biblioteca estándar de Python para análisis y manipulación de datos.
# La usamos para crear y manejar estructuras tabulares llamadas DataFrames.
import pandas as pd
# Importamos soundfile, una biblioteca especializada en leer y escribir archivos de audio.
# Nos permitirá extraer la información técnica de los audios sin tener que cargarlos completamente en memoria.
import soundfile as sf

# Importamos la función 'display' específica para entornos como Jupyter Notebooks o Google Colab.
# Permite renderizar y mostrar DataFrames (tablas) de forma mucho más limpia y visual que un simple 'print'.
from IPython.display import display

# Buscamos todos los archivos .flac dentro de la carpeta del corpus de forma recursiva (.rglob).
# Usamos sorted() para ordenar alfabéticamente la lista de rutas obtenidas y mantener la consistencia.
archivos_audio = sorted(carpeta_corpus.rglob("*.flac"))

# Inicializamos dos listas vacías. 
# Una para guardar los metadatos extraídos con éxito y otra para registrar los archivos que fallen.
registros_audio = []
errores_lectura = []

# Iniciamos un bucle para recorrer cada ruta de archivo de audio encontrada.
for ruta_audio in archivos_audio:
    # Usamos un bloque try-except porque leer archivos físicos siempre es propenso a errores 
    # (archivos corruptos, permisos denegados, etc.).
    try:
        # sf.info() lee exclusivamente las cabeceras (metadatos) del archivo de audio, lo cual es muy rápido.
        # Convertimos 'ruta_audio' (objeto Path) a string puro porque soundfile lo requiere así.
        informacion = sf.info(str(ruta_audio))

        # Creamos un diccionario recolectando toda la información que nos interesa del archivo y su estructura.
        registro = {
            "archivo": ruta_audio.name, # El nombre del archivo con su extensión (ej. 1234-5678-0001.flac)
            # La jerarquía de LibriSpeech es: locutor/capítulo/archivo.flac
            # Por lo tanto, el "abuelo" de la ruta del archivo es el ID del locutor.
            "locutor": ruta_audio.parent.parent.name, 
            # El "padre" de la ruta del archivo es el ID del capítulo del audiolibro.
            "capítulo": ruta_audio.parent.name,
            "ruta": str(ruta_audio), # La ruta absoluta en el disco para poder cargar el audio más adelante
            "duracion_segundos": informacion.duration, # Duración en segundos
            "frecuencia_muestreo": informacion.samplerate, # Tasa de muestreo (ej. 16000 Hz)
            "canales": informacion.channels, # Número de canales (1 para Mono, 2 para Estéreo)
            "numero_muestras":informacion.frames, # Cantidad total de puntos de datos en el audio
            "formato": informacion.format, # Formato contenedor (ej. FLAC)
            "subtipo": informacion.subtype # Subtipo de codificación de los datos
        }

        # Añadimos nuestro diccionario con los datos a la lista de registros exitosos.
        registros_audio.append(registro)
        
    # Si ocurre un error del sistema operativo (OSError) o de ejecución (RuntimeError) al leer el archivo:
    except (OSError, RuntimeError) as error:
        # Creamos un diccionario almacenando la ruta que falló y el mensaje del error exacto.
        registro_error = {
            "archivo": str(ruta_audio),
            "error": str(error)
        }

        # Añadimos el registro fallido a nuestra lista de errores.
        errores_lectura.append(registro_error)

# Convertimos la lista de diccionarios exitosos en un DataFrame (una tabla estructurada) usando pandas.
inventario_audio = pd.DataFrame(registros_audio)
# Hacemos lo mismo con la lista de errores, especificando explícitamente el nombre de las columnas.
tabla_errores = pd.DataFrame(
    errores_lectura,
    columns=["archivo","error"])

# Imprimimos un resumen cuantitativo en la consola.
print("Archivos encontrados:",len(archivos_audio))
print("Metadatos leídos:", len(inventario_audio))
print("Errores de lectura:", len(tabla_errores))

# Mostramos en pantalla las primeras 5 filas (.head()) de nuestra tabla principal de audios
# usando la función 'display' para que se vea con un formato agradable de tabla en el notebook.
display(inventario_audio.head())

# Si nuestra tabla de errores NO está vacía, la mostramos en pantalla para analizar qué falló.
if not tabla_errores.empty:
    display(tabla_errores)

# Validación final crítica: si la tabla principal está vacía, significa que falló la lectura 
# de absolutamente todos los archivos. Lanzamos un error deteniendo la ejecución.
if inventario_audio.empty:
    raise ValueError("No se pudieron leer los metadatos de ninguna grabación.")

# 7. Resumir la duración y los formatos del corpus

In [ ]:
# Obtenemos la cantidad total de grabaciones midiendo la longitud (número de filas) 
# de nuestro DataFrame 'inventario_audio'.
cantidad_grabaciones = len(inventario_audio)

# Calculamos el número de locutores únicos.
# Seleccionamos la columna "locutor" del DataFrame y usamos .nunique() (Number of Unique),
# que cuenta cuántos valores diferentes existen en esa columna, sin contar duplicados.
cantidad_locutores = inventario_audio["locutor"].nunique()

# Calculamos la duración total del dataset en segundos.
# Seleccionamos la columna "duracion_segundos" y aplicamos .sum() para sumar todos sus valores.
duracion_total_segundos = inventario_audio["duracion_segundos"].sum()

# Convertimos la duración total de segundos a minutos dividiendo entre 60.
duracion_total_minutos = duracion_total_segundos / 60
# Convertimos la duración de minutos a horas dividiendo nuevamente entre 60.
duracion_total_horas = duracion_total_minutos/60

# Imprimimos en consola el resumen general del corpus.
print("Locutores:", cantidad_locutores)
print("Grabaciones:", cantidad_grabaciones)
# Usamos f-strings para formatear la salida mostrando solo dos decimales (.2f).
print(f"Duración total en minutos: {duracion_total_minutos:.2f}")
print(f"Duración total en horas: {duracion_total_horas:.2f}")

# El método .describe() de pandas genera automáticamente estadísticas descriptivas 
# de una columna numérica (conteo, media, desviación estándar, mínimo, máximo y cuartiles).
estadisticas_duracion = inventario_audio["duracion_segundos"].describe()

print("\nEstadísticas de duración por grabación, en segundos:")
# Como .describe() devuelve un objeto de tipo Series, usamos .to_frame() para convertirlo 
# en un DataFrame de una sola columna (nombrada "duracion_segundos").
# Luego, usamos display() para mostrarlo con el formato de tabla estilizada del notebook.
display(estadisticas_duracion.to_frame(name="duracion_segundos"))

# Definimos una lista con los nombres de las columnas que contienen la información técnica del formato.
columnas_formato = [
    "frecuencia_muestreo",
    "canales",
    "formato",
    "subtipo"
]

# Agrupamos los datos del inventario basándonos en estas 4 columnas técnicas.
# .groupby() junta todas las filas que tengan exactamente la misma combinación de estos 4 valores.
grupos_formato = inventario_audio.groupby(columnas_formato)

# Usamos .size() sobre los grupos creados para contar cuántas grabaciones (filas) 
# existen dentro de cada grupo (es decir, cuántos audios comparten el mismo formato).
# Esto devuelve una Serie de pandas con un índice múltiple.
resumen_formatos = grupos_formato.size()

# .reset_index() convierte el índice múltiple nuevamente en columnas normales de un DataFrame.
# El parámetro name="grabaciones" asigna ese nombre a la nueva columna que contiene los conteos 
# calculados por .size().
resumen_formatos = resumen_formatos.reset_index(name = "grabaciones")

print("\nFormatos presentes en el corpus:")
# Mostramos la tabla resumen final donde se ven las combinaciones de formatos y cuántos audios hay de cada una.
display(resumen_formatos)

# 22. Extraer los MFCC de todas las grabaciones

In [ ]:
from time import perf_counter  # Importa el cronómetro de alta precisión para medir tiempos

mfcc_por_archivo = {}  # Diccionario: nombre de archivo -> matriz MFCC completa (tramas x coeficientes)
registros_caracteristicas = []  # Lista de diccionarios; cada uno será una fila del resumen por grabación

inicio_extraccion = perf_counter()  # Marca el instante en que inicia la extracción

for contador, registro in enumerate(  # Recorre cada grabación numerándola desde 1
    datos_particionados.itertuples(index=False),  # Itera las filas del DataFrame como tuplas, sin el índice
    start=1):  # El contador empieza en 1

    matriz = extraer_mfcc(registro.ruta)  # Lee el audio de esa ruta y calcula su matriz MFCC (una fila por trama)

    mfcc_por_archivo[registro.archivo] = matriz  # Guarda la matriz completa para usarla después (por ejemplo, con los GMM)

    medias = matriz.mean(axis=0)  # Media de cada coeficiente MFCC a lo largo de todas las tramas
    desviaciones = matriz.std(axis=0, ddof=0)  # Desviación estándar poblacional (ddof=0) de cada coeficiente

    resumen = {  # Diccionario que será la fila de esta grabación
        "archivo": registro.archivo,  # Nombre del archivo de audio
        "locutor": registro.locutor,  # Locutor al que pertenece la grabación
        "particion": registro.particion,  # Partición a la que pertenece (entrenamiento, validación o prueba)
        "numero_tramas": matriz.shape[0]}  # Cantidad de tramas (número de filas de la matriz MFCC)

    for indice in range(numero_mfcc):  # Recorre cada coeficiente MFCC (0, 1, 2, ...)
        resumen[f"media_mfcc_{indice}"] = float(medias[indice])  # Agrega la media del coeficiente como columna
        resumen[f"desviacion_mfcc_{indice}"] = float(desviaciones[indice])  # Agrega la desviación del coeficiente como columna

    registros_caracteristicas.append(resumen)  # Añade la fila completa a la lista (una vez por grabación)

    if contador % 500 == 0:  # Cada 500 grabaciones procesadas...
        print("Grabaciones procesadas: ", contador)  # ...imprime el avance

# ---- Fuera del ciclo: ya se procesaron todas las grabaciones ----

caracteristicas_audio = pd.DataFrame(registros_caracteristicas)  # Convierte la lista de resúmenes en una tabla

columnas_caracteristicas = []  # Lista con los nombres de las columnas que serán las características numéricas

for indice in range(numero_mfcc):  # Recorre cada coeficiente MFCC
    columnas_caracteristicas.append(f"media_mfcc_{indice}")  # Agrega el nombre de la columna de la media

for indice in range(numero_mfcc):  # Recorre otra vez cada coeficiente MFCC
    columnas_caracteristicas.append(f"desviacion_mfcc_{indice}")  # Agrega el nombre de la columna de la desviación

if len(mfcc_por_archivo) != len(datos_particionados):  # Comprueba que haya una matriz por cada grabación del inventario
    raise ValueError(  # Si no coinciden, detiene el programa con un error
        "El número de matrices no coincide con el inventario."  # Mensaje que explica el problema
    )  # Cierra la llamada a ValueError

valores_caracteristicas = caracteristicas_audio[  # Selecciona de la tabla solo las columnas de características
    columnas_caracteristicas].to_numpy()  # Las convierte en un arreglo de NumPy (grabaciones x características)

if not np.isfinite(valores_caracteristicas).all():  # Verifica que no haya NaN ni valores infinitos
    raise ValueError(  # Si los hay, detiene el programa con un error
        "Las características contienen valores no finitos.")  # Mensaje que explica el problema

tiempo_extraccion = perf_counter() - inicio_extraccion  # Segundos totales que tomó todo el proceso

print("\nMatrices MFCC", len(mfcc_por_archivo))  # Muestra cuántas matrices MFCC se guardaron
print("Grabaciones resumidas: ", len(caracteristicas_audio))  # Muestra cuántas filas tiene la tabla de resumen
print("Características por grabación: ", len(columnas_caracteristicas))  # Muestra cuántas características tiene cada grabación (medias + desviaciones)
print(f"Tiempo de extracción: {tiempo_extraccion:.2f}segundos")  # Muestra el tiempo total con 2 decimales

# 23. Preparar y escalar las características para k-NN

In [ ]:
from sklearn.preprocessing import StandardScaler  # Importa el escalador que estandariza (media 0, desviación 1)

caracteristicas_entrenamiento = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "entrenamiento"].copy()  # Se queda con las grabaciones de entrenamiento y hace una copia independiente

caracteristicas_validacion = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "validacion"].copy()  # Se queda con las grabaciones de validación y hace una copia independiente

caracteristicas_prueba = caracteristicas_audio.loc[  # Filtra la tabla completa con una condición sobre las filas
    caracteristicas_audio["particion"] == "prueba"].copy()  # Se queda con las grabaciones de prueba y hace una copia independiente

x_entrenamiento = caracteristicas_entrenamiento[  # Selecciona de la partición de entrenamiento...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

x_validacion = caracteristicas_validacion[  # Selecciona de la partición de validación...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

x_prueba = caracteristicas_prueba[  # Selecciona de la partición de prueba...
    columnas_caracteristicas].to_numpy()  # ...solo las columnas de características y las convierte en arreglo de NumPy

y_entrenamiento = caracteristicas_entrenamiento["locutor"].to_numpy()  # Etiquetas (locutor) de entrenamiento como arreglo

y_validacion = caracteristicas_validacion["locutor"].to_numpy()  # Etiquetas (locutor) de validación como arreglo

y_prueba = caracteristicas_prueba["locutor"].to_numpy()  # Etiquetas (locutor) de prueba como arreglo

escalador_knn = StandardScaler()  # Crea el escalador, todavía sin ajustar

x_entrenamiento_escalado = escalador_knn.fit_transform(x_entrenamiento)  # Calcula media y desviación SOLO con entrenamiento y lo escala

x_validacion_escalado = escalador_knn.transform(x_validacion)  # Escala validación con la media y desviación de entrenamiento

x_prueba_escalado = escalador_knn.transform(x_prueba)  # Escala prueba con la media y desviación de entrenamiento

for matri in [  # Recorre las tres matrices escaladas
    x_entrenamiento_escalado,  # Matriz de entrenamiento escalada
    x_validacion_escalado,  # Matriz de validación escalada
    x_prueba_escalado]:  # Matriz de prueba escalada

    if not np.isfinite(matri).all():  # Verifica que no haya NaN ni valores infinitos
        raise ValueError("Las características escaladas contienen valores no finitos.")  # Si los hay, detiene el programa con un error

print("Entrenamiento: ", x_entrenamiento_escalado.shape)  # Muestra (grabaciones, características) de entrenamiento
print("Validación: ", x_validacion_escalado.shape)  # Muestra (grabaciones, características) de validación
print("Prueba: ", x_prueba_escalado.shape)  # Muestra (grabaciones, características) de prueba

print("Locutores en entrenamiento: ", len(np.unique(y_entrenamiento)))  # Cuenta cuántos locutores distintos hay en entrenamiento

print("Grabaciones usadas para ajustar el escalador: ", escalador_knn.n_samples_seen_)  # Muestra cuántas muestras vio el escalador al ajustarse


# 24. Comparar configuraciones de k-NN en validación

In [ ]:
from sklearn.neighbors import KNeighborsClassifier  # Importa el clasificador de k vecinos más cercanos
from sklearn.metrics import accuracy_score, f1_score  # Importa las métricas de exactitud y F1

opciones_vecinos = [1, 3, 5, 7, 9, 21]  # Valores de k (número de vecinos) que se van a probar
opciones_pesos = ["uniform", "distance"]  # "uniform": todos los vecinos pesan igual; "distance": los más cercanos pesan más
opciones_p = [1, 2]  # Parámetro p de Minkowski: 1 = distancia Manhattan, 2 = distancia Euclidiana

resultados_knn = []  # Lista donde se guardan las métricas de cada combinación probada

mejor_modelo_knn = None  # Aquí se guardará el mejor modelo encontrado (aún no hay ninguno)
mejor_puntuacion_knn = (-1.0, -1.0)  # Mejor (F1 macro, exactitud) hasta ahora; empieza en -1 para que cualquier resultado lo supere
mejores_parametros_knn = None  # Aquí se guardarán los hiperparámetros del mejor modelo

for vecinos in opciones_vecinos:  # Ciclo externo: prueba cada valor de k
    for pesos in opciones_pesos:  # Ciclo intermedio: prueba cada tipo de ponderación
        for valor_p in opciones_p:  # Ciclo interno: prueba cada valor de p

            modelo = KNeighborsClassifier(  # Crea un clasificador KNN con la combinación actual
                n_neighbors=vecinos,  # Número de vecinos a considerar
                weights=pesos,  # Tipo de ponderación de los vecinos
                metric="minkowski",  # Métrica de distancia (generaliza Manhattan y Euclidiana)
                p=valor_p,  # Potencia de la distancia de Minkowski
                algorithm="brute",  # Búsqueda por fuerza bruta: compara contra todas las muestras
                n_jobs=-1)  # Usa todos los núcleos del procesador

            modelo.fit(x_entrenamiento_escalado, y_entrenamiento)  # "Entrena" el modelo (KNN solo memoriza los datos de entrenamiento)

            predicciones = modelo.predict(x_validacion_escalado)  # Predice el locutor de cada grabación de validación

            exactitud = accuracy_score(y_validacion, predicciones)  # Proporción de predicciones correctas

            f1_macro = f1_score(  # Calcula el F1-score
                y_validacion,  # Etiquetas reales
                predicciones,  # Etiquetas predichas
                average="macro",  # Promedia el F1 de cada locutor dándoles el mismo peso
                zero_division=0)  # Si hay división entre cero, asigna 0 en lugar de advertencia

            resultados_knn.append({  # Guarda el resultado de esta combinación
                "vecinos": vecinos,  # Valor de k usado
                "pesos": pesos,  # Ponderación usada
                "p": valor_p,  # Valor de p usado
                "exactitud_validacion": exactitud,  # Exactitud obtenida en validación
                "f1_macro_validacion": f1_macro})  # F1 macro obtenido en validación

            puntuacion = (f1_macro, exactitud)  # Tupla de puntuación: el F1 macro manda y la exactitud desempata

            if puntuacion > mejor_puntuacion_knn:  # Compara las tuplas elemento por elemento (primero F1, luego exactitud)
                mejor_puntuacion_knn = puntuacion  # Actualiza la mejor puntuación
                mejor_modelo_knn = modelo  # Guarda este modelo como el mejor

                mejores_parametros_knn = {  # Guarda los hiperparámetros del mejor modelo
                    "n_neighbors": vecinos,  # Mejor valor de k
                    "weights": pesos,  # Mejor ponderación
                    "p": valor_p}  # Mejor valor de p

# ---- Fuera de los ciclos: ya se probaron todas las combinaciones ----

tabla_resultados_knn = pd.DataFrame(resultados_knn)  # Convierte la lista de resultados en una tabla

tabla_resultados_knn = tabla_resultados_knn.sort_values(  # Ordena la tabla de resultados
    by=["f1_macro_validacion", "exactitud_validacion"],  # Primero por F1 macro y, en empate, por exactitud
    ascending=False,  # De mayor a menor (la mejor combinación queda arriba)
    kind="stable").reset_index(drop=True)  # Orden estable; reinicia el índice descartando el anterior

print("Configuraciones evaluadas:", len(tabla_resultados_knn))  # Muestra cuántas combinaciones se probaron

print(tabla_resultados_knn.to_string(  # Imprime la tabla completa como texto
    index=False,  # Sin mostrar el índice
    float_format="{:.4f}".format))  # Los decimales se muestran con 4 dígitos

print("\nMejores parámetros: ", mejores_parametros_knn)  # Muestra los hiperparámetros ganadores
print(f"F1 macro en validación: {mejor_puntuacion_knn[0]:.4f}")  # Muestra el F1 macro del mejor modelo con 4 decimales
print(f"Exactitud en validación: {mejor_puntuacion_knn[1]:.4f}")  # Muestra la exactitud del mejor modelo con 4 decimales


# 25. Identificar los errores de k-NN en validación

In [ ]:
predicciones_validacion_knn = mejor_modelo_knn.predict(  # Usa el mejor modelo KNN para predecir...
    x_validacion_escalado)  # ...el locutor de cada grabación de validación

revision_validacion_knn = caracteristicas_validacion[  # Toma la tabla de características de validación...
    ["archivo", "locutor"]].copy()  # ...y se queda solo con el archivo y el locutor real, en una copia independiente

revision_validacion_knn["locutor_predicho"] = (predicciones_validacion_knn)  # Agrega la columna con el locutor predicho por el modelo

revision_validacion_knn["acierto"] = (  # Agrega una columna booleana de acierto
    revision_validacion_knn["locutor"] == revision_validacion_knn["locutor_predicho"])  # True si el locutor real coincide con el predicho

revision_validacion_knn = revision_validacion_knn.merge(  # Une esta tabla con otra por una columna en común
    datos_validacion[["archivo", "duracion_segundos"]],  # Tabla con el nombre del archivo y la duración de cada audio
    on="archivo",  # La unión se hace por el nombre del archivo
    how="left",  # Conserva todas las filas de la tabla de revisión
    validate="one_to_one")  # Verifica que cada archivo aparezca una sola vez en ambas tablas

revision_validacion_knn["capitulo"] = (  # Agrega la columna del capítulo
    revision_validacion_knn["archivo"].str.split("-").str[1])  # Separa el nombre del archivo por guiones y toma el segundo elemento

errores_validacion_knn = revision_validacion_knn.loc[  # Filtra la tabla de revisión
    ~revision_validacion_knn["acierto"]].copy()  # Se queda solo con las filas donde NO hubo acierto (errores), en una copia

cantidad_aciertos = int(revision_validacion_knn["acierto"].sum())  # Suma los True (cada uno vale 1) para contar los aciertos

print("Grabaciones evaluadas:", len(revision_validacion_knn))  # Muestra el total de grabaciones evaluadas
print("Cantidad de aciertos: ", cantidad_aciertos)  # Muestra cuántas se clasificaron bien
print("Cantidad de errores: ", len(errores_validacion_knn))  # Muestra cuántas se clasificaron mal

columnas_revision_knn = [  # Lista de columnas que se mostrarán en la tabla de errores
    "archivo",  # Nombre del archivo de audio
    "locutor",  # Locutor real
    "locutor_predicho",  # Locutor que predijo el modelo
    "capitulo",  # Capítulo al que pertenece la grabación
    "duracion_segundos"]  # Duración del audio en segundos

if not errores_validacion_knn.empty:  # Si hubo al menos un error...
    print(errores_validacion_knn[columnas_revision_knn].to_string(  # ...imprime la tabla de errores solo con esas columnas
        index=False,  # Sin mostrar el índice
        float_format="{:.2f}".format))  # Los decimales se muestran con 2 dígitos
else:  # Si no hubo ningún error...
    print("No hubo errores en validación.")  # ...muestra este mensaje



# 26. Agrupar las tramas de entrenamiento para GMM

In [ ]:
tramas_entrenamiento_por_locutor = {}  # Diccionario: locutor -> matriz con todas sus tramas MFCC de entrenamiento
registros_tramas_gmm = []  # Lista de diccionarios; cada uno será una fila del resumen por locutor

for locutor, grupo in datos_entrenamiento.groupby("locutor", sort=True):  # Agrupa las grabaciones por locutor (ordenado) y recorre cada grupo

    matrices_locutor = []  # Lista temporal con las matrices MFCC de las grabaciones de este locutor

    for archivo in grupo["archivo"]:  # Recorre los archivos de audio de este locutor
        matrices_locutor.append(  # Agrega a la lista...
            mfcc_por_archivo[archivo])  # ...la matriz MFCC de ese archivo, buscándola por su nombre

    tramas_locutor = np.concatenate(matrices_locutor, axis=0)  # Apila todas las matrices una debajo de otra (une las tramas de todas sus grabaciones)

    if tramas_locutor.ndim != 2:  # Verifica que el resultado sea una matriz de 2 dimensiones (tramas x coeficientes)
        raise ValueError("Se esperaba una matriz bidimensional.")  # Si no, detiene el programa con un error

    if tramas_locutor.shape[1] != numero_mfcc:  # Verifica que el número de columnas sea igual al número de coeficientes MFCC
        raise ValueError("El número de MFCC no coincide.")  # Si no, detiene el programa con un error

    if not np.isfinite(tramas_locutor).all():  # Verifica que no haya NaN ni valores infinitos
        raise ValueError("Las tramas contienen valores no finitos.")  # Si los hay, detiene el programa con un error

    tramas_entrenamiento_por_locutor[locutor] = (tramas_locutor)  # Guarda las tramas del locutor en el diccionario

    registros_tramas_gmm.append({  # Agrega una fila de resumen para este locutor
        "locutor": locutor,  # Identificador del locutor
        "grabaciones_entrenamiento": len(grupo),  # Cantidad de grabaciones de entrenamiento que tiene
        "tramas_entrenamiento": tramas_locutor.shape[0]})  # Cantidad total de tramas (número de filas de la matriz)

# ---- Fuera del ciclo: ya se procesaron todos los locutores ----

resumen_tramas_gmm = pd.DataFrame(registros_tramas_gmm)  # Convierte la lista de resúmenes en una tabla

total_tramas_esperadas = sum(mfcc_por_archivo[archivo].shape[0]  # Suma el número de tramas de cada matriz MFCC...
                             for archivo in datos_entrenamiento["archivo"])  # ...recorriendo todos los archivos de entrenamiento

total_tramas_agrupadas = int(resumen_tramas_gmm["tramas_entrenamiento"].sum())  # Suma las tramas ya agrupadas por locutor, como entero

if total_tramas_agrupadas != total_tramas_esperadas:  # Comprueba que no se haya perdido ni duplicado ninguna trama
    raise ValueError("El total de tramas no coincide.")  # Si no coinciden, detiene el programa con un error

print("Locutores esperados:",  # Muestra cuántos locutores...
      len(tramas_entrenamiento_por_locutor))  # ...hay en el diccionario
print("Total de tramas de entrenamiento: ", total_tramas_agrupadas)  # Muestra el total de tramas de entrenamiento
print("Mínimo de tramas por locutor:",  # Muestra el menor número de tramas que tiene un locutor...
      resumen_tramas_gmm["tramas_entrenamiento"].min())  # ...calculado sobre la tabla de resumen
print("Máximo de tramas por locutor:",  # Muestra el mayor número de tramas que tiene un locutor...
      resumen_tramas_gmm["tramas_entrenamiento"].max())  # ...calculado sobre la tabla de resumen
print("MFCC por trama:", numero_mfcc)  # Muestra cuántos coeficientes MFCC tiene cada trama





# 27. Definir las configuraciones de GMM

In [ ]:
from sklearn.mixture import GaussianMixture  # Importa el modelo de mezclas gaussianas (GMM)

opciones_componentes_gmm = [8, 16, 32]  # Números de componentes gaussianas que se van a comparar

parametros_base_gmm = {  # Diccionario con los hiperparámetros comunes a todos los modelos
    "covariance_type": "diag",  # Covarianza diagonal: más rápida y con menos parámetros que la completa
    "tol": 2e-3,  # Tolerancia: el EM se detiene cuando la mejora es menor a este valor
    "reg_covar": 1e-6,  # Valor pequeño que se suma a la covarianza para evitar inestabilidad numérica
    "max_iter": 200,  # Máximo de iteraciones del algoritmo EM
    "n_init": 2,  # Número de inicializaciones; se conserva la mejor
    "init_params": "kmeans",  # Los parámetros iniciales se calculan con K-means
    "random_state": semilla}  # Semilla aleatoria para que los resultados sean reproducibles

minimo_tramas_gmm = min(matriz.shape[0]  # Obtiene el menor número de tramas...
                        for matriz in tramas_entrenamiento_por_locutor.values())  # ...entre las matrices de todos los locutores

if max(opciones_componentes_gmm) > minimo_tramas_gmm:  # Comprueba si el mayor número de componentes supera las tramas del locutor con menos datos
    raise ValueError("Hay más componentes que tramas disponibles.")  # Si es así, detiene el programa con un error

print("Componentes por comparar:", opciones_componentes_gmm)  # Muestra las cantidades de componentes a probar
print("Tipo de covarianza:", parametros_base_gmm["covariance_type"])  # Muestra el tipo de covarianza elegido
print("Máximo de iteraciones: ", parametros_base_gmm["max_iter"])  # Muestra el máximo de iteraciones del EM
print("Inicializacion por modelo:", parametros_base_gmm["n_init"])  # Muestra cuántas inicializaciones usa cada modelo
print("Semilla:", parametros_base_gmm["random_state"])  # Muestra la semilla aleatoria
print("Total de modelos por entrenar:",  # Muestra cuántos modelos se entrenarán en total...
      len(tramas_entrenamiento_por_locutor) * len(opciones_componentes_gmm))  # ...(número de locutores x número de configuraciones)

# 28. Entrenar y evaluar los GMM en validación

In [ ]:
modelos_gmm_por_configuracion = {}  # Diccionario: guarda, por cada número de componentes, los modelos de todos los locutores
resultados_gmm = []  # Lista donde se guardan las métricas de validación de cada configuración
registros_entrenamiento_gmm = []  # Lista donde se guarda el detalle del entrenamiento de cada modelo individual

locutores_gmm = sorted(tramas_entrenamiento_por_locutor.keys())  # Lista ordenada con los identificadores de los locutores
etiquetas_locutores_gmm = np.array(locutores_gmm)  # Mismo listado como arreglo de NumPy para indexarlo con argmax
y_validacion_gmm = datos_validacion["locutor"].to_numpy()  # Etiquetas reales (locutor) de las grabaciones de validación

for componentes in opciones_componentes_gmm:  # Prueba cada cantidad de componentes gaussianas candidata
    print(f"\nConfiguración: {componentes} componentes", flush=True)  # Informa qué configuración se está procesando

    modelos_actuales = {}  # Diccionario temporal: locutor -> modelo GMM de esta configuración
    inicio_entrenamiento = perf_counter()  # Marca el instante en que inicia el entrenamiento

    for contador, locutor in enumerate(locutores_gmm, start=1):  # Recorre los locutores numerándolos desde 1
        modelo = GaussianMixture(  # Crea un GMM vacío para este locutor
            n_components=componentes,  # Número de gaussianas de la mezcla
            **parametros_base_gmm)  # Desempaqueta el resto de hiperparámetros comunes (tipo de covarianza, iteraciones, etc.)
        modelo.fit(tramas_entrenamiento_por_locutor[locutor])  # Entrena el GMM con las tramas MFCC de ese locutor

        modelos_actuales[locutor] = modelo  # Guarda el modelo entrenado asociado a su locutor
        registros_entrenamiento_gmm.append({  # Agrega un registro con los datos del entrenamiento
            "componentes": componentes,  # Configuración usada
            "locutor": locutor,  # Locutor al que pertenece el modelo
            "convergio": modelo.converged_,  # True si el algoritmo EM convergió
            "iteraciones": modelo.n_iter_})  # Número de iteraciones que necesitó el EM

        if contador % 5 == 0:  # Cada 5 modelos entrenados...
            print(  # ...imprime el avance
                f"Modelos entrenados: {contador}/{len(locutores_gmm)}",  # Muestra "entrenados/total"
                flush=True)  # Fuerza la salida inmediata en pantalla

    tiempo_entrenamiento = perf_counter() - inicio_entrenamiento  # Segundos totales que tomó entrenar esta configuración

    modelos_gmm_por_configuracion[componentes] = modelos_actuales  # Guarda todos los modelos de esta configuración

    inicio_validacion = perf_counter()  # Marca el instante en que inicia la validación
    predicciones_actuales = []  # Lista donde se guardan los locutores predichos

    for contador, registro in enumerate(  # Recorre cada grabación de validación numerándola desde 1
            datos_validacion.itertuples(index=False),  # Itera las filas del DataFrame como tuplas, sin el índice
            start=1):  # El contador empieza en 1
        matriz = mfcc_por_archivo[registro.archivo]  # Obtiene la matriz de MFCC del archivo de audio actual

        puntuaciones = np.array([  # Arreglo con una puntuación por cada locutor
            modelos_actuales[locutor].score(matriz)  # Log-verosimilitud promedio de la grabación bajo el modelo de ese locutor
            for locutor in locutores_gmm])  # Se calcula para todos los locutores, en el mismo orden

        if not np.isfinite(puntuaciones).all():  # Verifica que ninguna puntuación sea NaN o infinita
            raise ValueError(  # Si hay alguna, detiene el programa con un error
                f"Puntuaciones no finitas en {registro.archivo}.")  # Indica en qué archivo ocurrió el problema

        indice_mejor = int(np.argmax(puntuaciones))  # Posición del locutor con la puntuación más alta
        predicciones_actuales.append(  # Guarda la predicción de esta grabación
            etiquetas_locutores_gmm[indice_mejor])  # Convierte la posición en el identificador del locutor

        if contador % 100 == 0:  # Cada 100 grabaciones evaluadas...
            print(  # ...imprime el avance
                f"Grabaciones de validación evaluadas: "  # Primera parte del mensaje
                f"{contador}/{len(datos_validacion)}",  # Muestra "evaluadas/total"
                flush=True)  # Fuerza la salida inmediata en pantalla

    tiempo_validacion = perf_counter() - inicio_validacion  # Segundos totales que tomó la validación

    exactitud = accuracy_score(  # Calcula la proporción de predicciones correctas
        y_validacion_gmm,  # Etiquetas reales
        predicciones_actuales)  # Etiquetas predichas por los GMM

    f1_macro = f1_score(  # Calcula el F1-score
        y_validacion_gmm,  # Etiquetas reales
        predicciones_actuales,  # Etiquetas predichas
        average="macro",  # Promedia el F1 de cada locutor dándoles el mismo peso
        zero_division=0)  # Si hay división entre cero, asigna 0 en lugar de advertencia

    modelos_sin_convergencia = sum(  # Cuenta cuántos modelos no convergieron
        not modelo.converged_  # True (1) cuando el modelo no convergió
        for modelo in modelos_actuales.values())  # Revisa todos los modelos de la configuración actual

    resultados_gmm.append({  # Agrega el resumen de esta configuración
        "componentes": componentes,  # Número de componentes probado
        "exactitud_validacion": exactitud,  # Exactitud obtenida en validación
        "f1_macro_validacion": f1_macro,  # F1 macro obtenido en validación
        "modelos_sin_convergencia": modelos_sin_convergencia,  # Cantidad de modelos que no convergieron
        "tiempo_entrenamiento_s": tiempo_entrenamiento,  # Tiempo de entrenamiento en segundos
        "tiempo_validacion_s": tiempo_validacion})  # Tiempo de validación en segundos

    print(f"F1 macro: {f1_macro:.4f} | "  # Muestra el F1 macro con 4 decimales
          f"Exactitud: {exactitud:.4f} | "  # Muestra la exactitud con 4 decimales
          f"Modelos sin convergencia: {modelos_sin_convergencia}",  # Muestra cuántos modelos no convergieron
          flush=True)  # Fuerza la salida inmediata en pantalla

# ---- Fuera del ciclo: ya se probaron todas las configuraciones ----

tabla_entrenamiento_gmm = pd.DataFrame(  # Convierte los registros de entrenamiento en una tabla
    registros_entrenamiento_gmm)  # Lista de diccionarios con el detalle de cada modelo

tabla_resultados_gmm = pd.DataFrame(resultados_gmm)  # Convierte los resultados por configuración en una tabla

tabla_resultados_gmm = tabla_resultados_gmm.sort_values(  # Ordena la tabla de resultados
    by=["f1_macro_validacion", "exactitud_validacion"],  # Primero por F1 macro y, en empate, por exactitud
    ascending=False,  # De mayor a menor (la mejor configuración queda arriba)
    kind="stable").reset_index(drop=True)  # Orden estable; reinicia el índice descartando el anterior

mejores_componentes_gmm = int(  # Número de componentes de la mejor configuración, como entero
    tabla_resultados_gmm.iloc[0]["componentes"])  # Toma la primera fila (la mejor) y su columna "componentes"

mejores_modelos_gmm = modelos_gmm_por_configuracion[  # Recupera los modelos de la mejor configuración
    mejores_componentes_gmm]  # Se busca por su número de componentes

print("\nResultados de GMM:")  # Título de la tabla de resultados
print(tabla_resultados_gmm.to_string(  # Imprime la tabla completa como texto
    index=False,  # Sin mostrar el índice
    float_format="{:.4f}".format))  # Los decimales se muestran con 4 dígitos

print("\nComponentes seleccionados:", mejores_componentes_gmm)  # Muestra cuántos componentes ganaron

# Representación digital de voz
* Frecuencia de muestreo: cantidad de muestras que se toman por unidad de tiempo para convertir la señal analógica en señal digital.
* Profundidad de bits: cantidad de bits que pueden mostrar color o amplitud en imágenes o sonidos. Indica el nivel de detalle y precisión al grabar o mostrar contenido digital.
* Canales: representación de una señal de audio en los procesos de grabación, reproducción o mezcla
* Cuantificación: proceso en el cual se asignan valores finitos, de un conjunto de niveles definidos previamente, a las muestras tomadas de la señal analógica.
* Teorema de Nyquist-Shannon: establece que para reconstruir una señal analógica continua en el tiempo sin perder información, la tasa de muestreo debe ser mayor al doble de su frecuencia máxima.

# Corpus de voz
LibriSpeech
* Características: conjunto de datos con grabaciones de audio libros de dominio público.
* Estructura: 1, 000 horas de grabaciones de voz en inglés de forma leída. Los audios estan muestreados a 16KHz y cuentan con transcripciones de texto alineadas.
* Aplicaciones: entrenamiento de modelos ASR, clasificación e identificación de hablantes, análisis de audio y emociones.
* Limitaciones: dominio exclusivo del habla leída, falta de ruido ambiental real, sesgo lingüístico y de acentos, ausencia de dialogo superpuesto y las transcirpciones están en texto plano, sin acentos y en minúsculas.

Common voice
* Características: base de datos multilingüe y de dominio público, los datos se obtienen de la participación voluntaria de las personas, reúne muestras de voz en cientos de idiomas. Mitigación de sesgos en la IA, sirve para entrenar modelos de aprendizaje automático.
* Estructura: frases cortas preestablecidas, respuestas libres e improvisadas a preguntas casuales, reflejan conversaciones desordenadas de la vida diaria.
* Aplicaciones: sistemas de reconocimiento automático del habla, modelo de lenguaje y transfer learning, sistemas de texto a voz
* Limitaciones: calidad del audio, distribución demográfica sesgada, metadatos incompletos, desequilibrio entre idiomas.

VoxCeleb
* Características:conjunto de datos a gran escala de audio y video diseñado para el reconocimiento de locutores y la biometría de voz en condiciones reales. Extraído de videos de YouTube, incluye ruido del mundo real, abarca múltiples etnias, acentos, edades y profesionales.
* Estructura: esta estructurado de forma jerárquica para facilitar el procesamiento tanto de audio como de video: nivel 1 (identidad), nivel 2 (video de origen), nivel 3 (fragmentos). Más de 1 millón de enunciados, más de 2000 horas de audio.
* Aplicaciones: verificación del locutor, identificación del locutor, conversión de texto a voz y clonación de voz, modelos multimodales.
* Limitaciones: sesgo demográfico y geográfico, sesgo de edad, calidad variable del audio, falsos positivos en la recolección.

# Librerías de procesamiento de audio

Librosa
* Funciones principales: carga de audio, representaciones espectrales y manipulación de señales.
* Características: es una librería para el procesamiento de audio y música. Proporciona los algoritmos y herramientas fundamentales necesarios para construir sistemas de recuperación de información musical.

SoundFile
* Funciones principales: funciones de lectura/escritura, procesamiento de bloques, los archivos de audio se pueden abrir como objetos, se pueden leer y escribir fragmentos de audio sin tocar el sistema de archivos.
* Características: puede leer y escribir archivos de audio

* Diferencias entre ambos: SoundFile se utiliza para leer y escribir archivos de audio, mientras que Librosa es una librería para el análisis musical y extracción de características de audio.